# Feature Engineering on High Volume For-Hire Vehicle (HVFHV) Trip Records Dataset:

In this notebook, we are mainly focusing on adding helpful columns in HVFHV dataset.

----

# Import Libraries:

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import * 
from pyspark.sql.functions import when, col, date_format, \
                                    unix_timestamp, to_date, hour, month
import os

In [2]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("feature_engineering_hvfhv")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.network.timeout", "600s")
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.rpc.askTimeout", "600s")
    .config("spark.driver.memory", "100G")
    .config("spark.executor.memory", "100G")
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.sql.debug.maxToStringFields", "1000")
    .getOrCreate()
)

24/08/17 19:25:01 WARN Utils: Your hostname, Cocos-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 172.16.33.67 instead (on interface en0)
24/08/17 19:25:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/17 19:25:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Read Files:

In [3]:
base_dir = "../data"

In [4]:
hvfhv_path = base_dir + '/curated/hvfhv_data/preprocessed_hvfhv_2'
hvfhv_sdf = spark.read.parquet(hvfhv_path)
hvfhv_sdf.show(5)

+-----------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+--------------+
|hvfhs_license_num|   request_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|wav_request_flag|wav_match_flag|
+-----------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+--------------+
|                0|2023-09-09 00:37:59|2023-09-09 00:45:24|2023-09-09 01:15:28|          39|         231|     20.85|    30.07|             

In [5]:
num_rows = hvfhv_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hvfhv_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

Number of rows: 100725779
Number of columns: 20


# Add New Columns:

Since as a driver, they concentrate on where I go could increase the revenue, so we are focusing on the pickup location.

Add new column `day_of_week` to indicate it is Monday or Tuesday and so on based on `pickup_datetime`:

In [6]:
hvfhv_sdf = hvfhv_sdf.withColumn("day_of_week", date_format("pickup_datetime", "EEEE"))
hvfhv_sdf.show(5)

+-----------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+--------------+-----------+
|hvfhs_license_num|   request_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|wav_request_flag|wav_match_flag|day_of_week|
+-----------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+--------------+-----------+
|                0|2023-09-09 00:37:59|2023-09-09 00:45:24|2023-09-09 01:15:28|          39|         23

Add new column `request_to_pickup_time` to indicate the waiting time in minutes for passengers after they request a HVFHV (calculated by `pickup_datetime` - `request_datetime`):

In [7]:
hvfhv_sdf = hvfhv_sdf.withColumn(
    "request_to_pickup_minutes",
    (unix_timestamp("pickup_datetime") - unix_timestamp("request_datetime")) / 60
)
hvfhv_sdf.show(5)

+-----------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+--------------+-----------+-------------------------+
|hvfhs_license_num|   request_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|wav_request_flag|wav_match_flag|day_of_week|request_to_pickup_minutes|
+-----------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+--------------+-----------+-------------------------+
|                0|2023-0

Add new column `trip_speed` (unit: miles per minutes) calculated by `trip_miles`/`trip_time` :

In [8]:
hvfhv_sdf = hvfhv_sdf.withColumn(
    "trip_speed",
    col("trip_miles") / col("trip_time")
)
hvfhv_sdf.show(5)

+-----------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+--------------+-----------+-------------------------+-------------------+
|hvfhs_license_num|   request_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|wav_request_flag|wav_match_flag|day_of_week|request_to_pickup_minutes|         trip_speed|
+-----------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+--------------+-----------+------------

Add new column `total_fare_amount` calculated by `base_passenger_fare` + `tolls` + `bcf` + `sales_tax` + `congestion_surcharge` + `airport_fee` + `tips`:

In [9]:
hvfhv_sdf = hvfhv_sdf.withColumn(
    "total_fare_amount",
    col("base_passenger_fare") + col("tolls") + col("bcf") + col("sales_tax") + 
    col("congestion_surcharge") + col("airport_fee") + col("tips")
)
hvfhv_sdf.show(5)

+-----------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+--------------+-----------+-------------------------+-------------------+------------------+
|hvfhs_license_num|   request_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|wav_request_flag|wav_match_flag|day_of_week|request_to_pickup_minutes|         trip_speed| total_fare_amount|
+-----------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+-

Add new column `total_revenue` calculated by `driver_pay` + `tips`:

In [10]:
hvfhv_sdf = hvfhv_sdf.withColumn(
    "total_revenue", col("driver_pay") + col("tips")
)
hvfhv_sdf.show(5)

+-----------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+--------------+-----------+-------------------------+-------------------+------------------+-------------+
|hvfhs_license_num|   request_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|wav_request_flag|wav_match_flag|day_of_week|request_to_pickup_minutes|         trip_speed| total_fare_amount|total_revenue|
+-----------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+--------

Extract the date and hour from the `pickup_datetime` and `dropoff_datetime` columns:


In [11]:
hvfhv_sdf = hvfhv_sdf.withColumn("pickup_date", to_date("pickup_datetime")) \
                     .withColumn("pickup_hour", hour("pickup_datetime")) \
                     .withColumn("dropoff_date", to_date("dropoff_datetime")) \
                     .withColumn("dropoff_hour", hour("dropoff_datetime")) \
                     .withColumn("month", month("pickup_datetime"))

hvfhv_sdf = hvfhv_sdf.drop("request_datetime", "pickup_datetime", "dropoff_datetime")
hvfhv_sdf.show(5)

+-----------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+--------------+-----------+-------------------------+-------------------+------------------+-------------+-----------+-----------+------------+------------+-----+
|hvfhs_license_num|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|wav_request_flag|wav_match_flag|day_of_week|request_to_pickup_minutes|         trip_speed| total_fare_amount|total_revenue|pickup_date|pickup_hour|dropoff_date|dropoff_hour|month|
+-----------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+--------------+-----------+--------------

In [12]:
# Check the shape the parquet file
num_rows = hvfhv_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hvfhv_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

Number of rows: 100725779
Number of columns: 27


# Save the Full HVFHV Dataset:

In [13]:
hvfhv_dir = base_dir + '/developed/merged_data'
file_name = 'full_hvfhv'
hvfhv_path = os.path.join(hvfhv_dir, file_name)
hvfhv_sdf.write.mode('overwrite').parquet(hvfhv_path)